In [1]:
import pandas as pd
import os

# Format data for graphics

Columns needed for the `parent_table` sheet:
- SOC_code
- Occupation
- AI_adaptation_score
- Employment

Columns needed fro the `child_table` sheet: 
- child_id (detailed SOC)
- Occupation
- AI_adaptation_score
- parent_id (general SOC found in parent_table)

Columns needed for the `scatterplot_data` sheet:
- it's actually the same as the `parent_table` so not going to create a whole other export for this!

In [2]:
# get the processed BLS+study data for our metro area
market_state = 'California'
market_state_abbr = 'CA'
market_metro = 'San Francisco-Oakland-Fremont, CA'
market_metro_abbr = 'sfc'

dtypes = {'series_id':str, 'year':int, 'period':str, 'areatype_code':str, 'state_code':str,
       'area_code':str, 'area_name':str, 'occupation_code':str, 'occupation_name':str,
       'employment':float, 'study_rating':float, 'footnote_codes':str, 'is_all_occupations':bool,
       'has_released_employment':bool, 'NEM Code':str, 'nem_merge':str,
       'ai_exposure_category':str, 'employment_category':str, 'soc_code_study':str}
oews_with_scores = pd.read_csv('../data/processed/bls_occ_employment_w_study_scores_human_rating_beta.csv', dtype=dtypes)

#oews_with_scores['SOC_code'] = oews_with_scores['occupation_code'].str[:2] + '-' + oews_with_scores['occupation_code'].str[2:4] + '.00'
#oews_with_scores['ai_exposure_category'] = oews_with_scores['ai_exposure_category'].astype(size_order)

if market_state == 'Connecticut':
    main_metro_scores = oews_with_scores[oews_with_scores['area_name'] == 'Connecticut']
else:
    main_metro_scores = oews_with_scores[oews_with_scores['area_name'] == market_metro]

# pull in the study data and reduce to just occupations that aren't in the oews processed data
study_rating = 'human_rating_beta'
occ_scores = pd.read_csv('https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/occ_level.csv')
occ_scores = occ_scores[['O*NET-SOC Code', 'Title', study_rating]]

print('Total occupations scored by the study:', len(occ_scores))

child_scores = occ_scores[~(occ_scores['O*NET-SOC Code'].isin(main_metro_scores['soc_code_study'].unique()))]
child_scores['parent_id'] = child_scores['O*NET-SOC Code'].str[:7]
child_scores = child_scores.rename(columns={study_rating: 'AI_adaptation_score',
                                                  'O*NET-SOC Code': 'child_id',
                                                  'Title': 'Occupation'})
child_scores = child_scores[['parent_id', 'child_id', 'Occupation', 'AI_adaptation_score']]

# check how many child scores we have and how many of those are for .00 codes (which are the ones 
# that are more likely to be parents themselves). usually it means that the main metro just didn't 
# have estimates for those occupations but do a little checky check on the BLS one-screen finder tool
# to confirm: https://data.bls.gov/oes/#/home
print('Total child scores:',len(child_scores))
print('Child scores that end with .00:',len(child_scores.loc[child_scores['child_id'].str.endswith('.00')]))
print('Sample child scores with .00:')
display(child_scores.loc[child_scores['child_id'].str.endswith('.00')].sample(10))

Total occupations scored by the study: 923
Total child scores: 302
Child scores that end with .00: 158
Sample child scores with .00:


,parent_id,child_id,Occupation,AI_adaptation_score
362,27-2091,27-2091.00,"Disc Jockeys, Except Radio",0.368421
3,11-1031,11-1031.00,Legislators,0.516667
318,25-2032,25-2032.00,"Career/Technical Education Teachers, Secondary...",0.333333
132,15-2021,15-2021.00,Mathematicians,0.690476
316,25-2023,25-2023.00,"Career/Technical Education Teachers, Middle Sc...",0.338710
860,51-9191,51-9191.00,Adhesive Bonding Machine Operators and Tenders,0.000000
869,51-9197,51-9197.00,Tire Builders,0.000000
381,29-1022,29-1022.00,Oral and Maxillofacial Surgeons,0.080000
212,19-2011,19-2011.00,Astronomers,0.421875
724,49-2097,49-2097.00,Audiovisual Equipment Installers and Repairers,0.095238


I've looked up a sample of these codes on the [BLS OES one-page finder tool](https://data.bls.gov/oes/#/home) and none have values for this metro.

In [3]:
# and now we remove those .00 records because we've confirmed that they're just not relevant to our market
#child_scores = child_scores.loc[~child_scores['child_id'].str.endswith('.00')] 
#print('Child scores after removing .00:', len(child_scores))

# just grab the parent columns we need
print(len(main_metro_scores))
main_metro_scores['SOC_code'] = main_metro_scores['soc_code_study'].str[:7]
parent_scores = main_metro_scores.rename(columns={'occupation_name': 'Occupation',
                                                  'employment': 'Employment',
                                                  'study_rating': 'AI_adaptation_score'})
parent_scores = parent_scores[['SOC_code', 'Occupation', 'Employment', 'AI_adaptation_score']].drop_duplicates()
print(len(parent_scores))

#Also removing any of the parent records without scores
parent_scores = parent_scores.loc[~parent_scores['AI_adaptation_score'].isna()]
print('Parent scores after removing records without scores:', len(parent_scores))


#make an output folder for our state if it doesn't already exist
os.makedirs(f'../data/output/{market_state_abbr.lower()}', exist_ok=True)

#and export the parent and child scores to csv for use in our graphics code
parent_scores.to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_parent_scores.csv', index=False)
child_scores.to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_child_scores.csv', index=False)

699
699
Parent scores after removing records without scores: 635
